# ChaoSSM — Phase 20: a direct generalized-synchronization diagnostic

Addresses **Reviewer 2, comment 2**: *provide direct synchronization diagnostics to complement the
conditional Lyapunov measurements — drive two identical reservoirs with the same input from
different initial states and report the asymptotic state discrepancy across λ and under trained
vs random-input protocols.*

This is the **auxiliary-system (replica) test** for generalized synchronization. Two copies of the
same frozen reservoir — identical `W_mix`, `r`, `γ`, `B` — are driven by the **same** input
sequence from **different** random initial states. If the reservoir possesses the echo state
property (its state is a function of the input history alone), the two trajectories converge and
the asymptotic discrepancy `‖h₁−h₂‖ → 0`; if the driven dynamics remain chaotic, the discrepancy
stays positive.

We sweep λ from −0.60 to +0.30 under the **random-input** protocol (B random, frozen), and repeat
at the three named regimes under the **trained-input** protocol (B trained on the forecast task,
then frozen). The paper claims the collapse past the chaos onset coincides with a discontinuous
loss of the echo state property; this figure is the direct measurement of that claim, and it
should mirror the conditional-λ result (−0.085 random → −0.710 trained).

Runtime: ~10–15 min on a T4; mostly forward passes. Resumable.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, field
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))

def setup_results_dir(project="ChaoSSM",phase="phase20_synchronization"):
    base=None
    try:
        from google.colab import drive
        try: drive.mount("/content/drive",force_remount=False)
        except Exception as e1:
            print(f"[warn] drive.mount failed ({type(e1).__name__}); retrying force_remount=True")
            drive.mount("/content/drive",force_remount=True)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"/content/{project}_results/{phase}") if os.path.isdir("/content") \
             else os.path.abspath(f"./{project}_results/{phase}")
        print("="*78)
        print(f"[WARNING] Google Drive NOT mounted ({type(e).__name__}). Results -> {base}")
        print("[WARNING] That is local disk, lost on runtime recycle. Run the rescue cell or download.")
        print("="*78)
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=600,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)

torch 2.11.0+cu130 | device: cuda
gpu: Tesla T4
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase20_synchronization


## Configuration

In [2]:
SMOKE_TEST=False

@dataclass
class Config:
    state_dim:int=64; d_model:int=64; r_cap_map:float=4.0; gamma_init:float=1.0
    calib_steps:int=200; calib_warm:int=60; lambda_tol:float=0.05
    seeds:int=5
    # random-input sweep: the seven-point Lyapunov axis of Phase 15, plus the three named regimes
    lambda_axis:tuple=(-0.60,-0.45,-0.30,-0.15,-0.02,0.15,0.30)
    named:dict=field(default_factory=lambda: dict(ordered=-0.30, edge=-0.02, deep=0.30))
    # synchronization measurement
    sync_T:int=600; sync_washout:int=400; n_pairs:int=8
    # trained-input protocol (forecast task, B+readout trained then frozen)
    fc_length:int=64; fc_bins:int=32; steps:int=1500; batch:int=64; lr:float=1e-3
    grad_clip:float=1.0; warmup_frac:float=0.05; lr_min_frac:float=0.1
CFG=Config()
if SMOKE_TEST:
    CFG.seeds=2; CFG.lambda_axis=(-0.30,-0.02,0.30); CFG.sync_T=120; CFG.sync_washout=80
    CFG.n_pairs=3; CFG.steps=40; CFG.named=dict(edge=-0.02)
    print("*** SMOKE TEST — results meaningless ***")
print("lambda axis:",CFG.lambda_axis,"| named regimes:",list(CFG.named))

lambda axis: (-0.6, -0.45, -0.3, -0.15, -0.02, 0.15, 0.3) | named regimes: ['ordered', 'edge', 'deep']


## Cell, calibration — verbatim from Phase 14/15

In [3]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

class ChaosCell(nn.Module):
    def __init__(self,cfg):
        super().__init__(); self.N=cfg.state_dim; self.r_cap=float(cfg.r_cap_map)
        self.w_r=nn.Parameter(torch.zeros(self.N))
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(cfg.gamma_init)))))
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad(): self.w_gamma.copy_(torch.tensor(float(math.log(math.expm1(max(val,1e-3))))))
    def phi(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)
    def step(self,h,x): return triangle_fold(self.phi(h)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze_recurrence(self):
        for p_ in [self.w_r,self.W_mix,self.w_gamma]: p_.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0):
    steps,warm,N=cfg.calib_steps,cfg.calib_warm,cfg.state_dim; dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z):
            r=cell.get_r(); u=z@cell.W_mix.T+r*z*(1-z); return triangle_fold(cell.get_gamma()*u)
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev); h=torch.clamp(h,0,1)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate_to(cell,cfg,target,seed=0):
    lo,hi=0.01,8.0; best=None; best_err=1e9
    for _ in range(20):
        mid=math.sqrt(lo*hi); cell.set_gamma(mid); lam=map_lyapunov(cell,cfg,seed=seed)
        err=abs(lam-target)
        if err<best_err: best_err=err; best=(mid,lam)
        if lam<target: lo=mid
        else: hi=mid
        if err<cfg.lambda_tol: break
    cell.set_gamma(best[0]); return best[0],best[1],bool(best_err<cfg.lambda_tol)

## The synchronization diagnostic

Two state trajectories share the cell and the input `u`, differ only in initial condition. The
normalized asymptotic discrepancy is the mean of `‖h₁−h₂‖ / ½(‖h₁‖+‖h₂‖)` over the post-washout
window, averaged over several input realizations and initial-state pairs. `D → 0` means the
reservoir synchronizes (echo state property holds); `D` bounded away from 0 means it does not.

In [4]:
@torch.no_grad()
def sync_discrepancy(cell, cfg, encode_fn, seed=0):
    dev=next(p.device for p in cell.parameters()); N=cfg.state_dim
    g=torch.Generator(device=dev); g.manual_seed(seed)
    Draw=[]
    for rep in range(cfg.n_pairs):
        u=torch.rand(cfg.sync_T,generator=g,device=dev)
        drive=encode_fn(u.view(1,-1)).squeeze(0)                 # (T, d_model)
        h1=torch.rand(N,generator=g,device=dev)                  # two different init states
        h2=torch.rand(N,generator=g,device=dev)
        disc=[]
        for t in range(cfg.sync_T):
            x=drive[t]
            h1=triangle_fold(cell.phi(h1)+x@cell.B.T)
            h2=triangle_fold(cell.phi(h2)+x@cell.B.T)
            if t>=cfg.sync_washout:
                num=torch.norm(h1-h2); den=0.5*(torch.norm(h1)+torch.norm(h2))+1e-9
                disc.append(float(num/den))
        Draw.append(float(np.mean(disc)))
    return float(np.mean(Draw)), float(np.std(Draw))

## Trained-input protocol: train B + readout on the forecast task, then freeze

In [5]:
def make_forecast(cfg,batch,device,gen):
    L=cfg.fc_length; r=3.9
    x0=0.1+0.8*torch.rand(batch,1,device=device,generator=gen); xs=[x0]
    for _ in range(L): xs.append(r*xs[-1]*(1-xs[-1]))
    X=torch.cat(xs,1); inp=X[:,:L]; nxt=X[:,1:L+1]
    tgt=torch.clamp((nxt*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp.float(),tgt

class TrainedInputModel(nn.Module):
    """Trained-input fair-test model: recurrence frozen, but B AND the encoder are TRAINED
    (this is the input-driven-synchronization condition from the paper's Table 2)."""
    def __init__(self,cfg,cell):
        super().__init__(); self.inp=nn.Linear(1,cfg.d_model); self.cell=cell
        self.ln=nn.LayerNorm(cfg.state_dim); self.readout=nn.Linear(cfg.state_dim,cfg.fc_bins)
        cell.freeze_recurrence()                 # recurrence frozen; B stays trainable
    def forward(self,x):
        e=self.inp(x.unsqueeze(-1)); B,T=e.shape[0],e.shape[1]
        h=self.cell.init_state(B,e.device,e.dtype); H=[]
        for t in range(T): h=self.cell.step(h,e[:,t]); H.append(h)
        return self.readout(self.ln(torch.stack(H,1)))

def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

def train_input(cfg,cell,seed):
    """Train B+encoder+readout on forecast; return the trained cell (recurrence untouched)."""
    model=TrainedInputModel(cfg,cell).to(DEVICE)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(params,lr=cfg.lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(cfg,cfg.steps))
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    diverged=False
    for step in range(cfg.steps):
        inp,tgt=make_forecast(cfg,cfg.batch,DEVICE,gen)
        lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1))
        opt.zero_grad(set_to_none=True); L.backward()
        torch.nn.utils.clip_grad_norm_(params,cfg.grad_clip); opt.step(); sched.step()
        if not math.isfinite(L.item()): diverged=True; break
    cell.freeze_input()                          # now freeze B for the sync measurement
    # the trained encoder is what the sync drive must use, so return it
    enc=model.inp
    for p_ in enc.parameters(): p_.requires_grad_(False)
    return cell, enc, diverged

## Run — random-input sweep across λ, then trained-input at the named regimes

In [6]:
FRESH_START=False
CKPT=os.path.join(RESULTS,"phase20_synchronization.csv")
COLS=["protocol","regime","target_lambda","seed","achieved_lambda","calib_ok","sync_D","sync_D_sd","train_diverged"]
if FRESH_START and os.path.exists(CKPT): os.replace(CKPT,CKPT+".bak")
if os.path.exists(CKPT):
    rows=pd.read_csv(CKPT).to_dict("records")
    done={(r["protocol"],str(r["regime"]),int(r["seed"])) for r in rows}
    print(f"[resume] {len(rows)} runs done")
else: rows=[]; done=set(); print("[resume] fresh start")
def _flush(): pd.DataFrame(rows)[COLS].to_csv(CKPT,index=False)

plan=[]
for lam in CFG.lambda_axis:
    for s in range(CFG.seeds): plan.append(("random-input",f"{lam:+.2f}",lam,s))
for name,lam in CFG.named.items():
    for s in range(CFG.seeds): plan.append(("trained-input",name,lam,s))
todo=[p for p in plan if (p[0],p[1],p[3]) not in done]
print(f"{len(todo)} runs to go\n"); t0=time.time()

for protocol,regime,target,sd in todo:
    torch.manual_seed(sd); np.random.seed(sd)
    cell=ChaosCell(CFG).to(DEVICE)
    _,lam,ok=calibrate_to(cell,CFG,target,seed=sd)
    if protocol=="random-input":
        cell.freeze_recurrence(); cell.freeze_input()
        enc=nn.Linear(1,CFG.d_model).to(DEVICE)
        for p_ in enc.parameters(): p_.requires_grad_(False)
        tdiv=False
    else:
        cell,enc,tdiv=train_input(CFG,cell,sd)      # B + encoder trained, recurrence intact
    encode_fn=lambda u,_e=enc: _e(u.unsqueeze(-1))
    D,Dsd=sync_discrepancy(cell,CFG,encode_fn,seed=sd)
    rows.append(dict(protocol=protocol,regime=regime,target_lambda=target,seed=sd,
                     achieved_lambda=lam,calib_ok=ok,sync_D=D,sync_D_sd=Dsd,
                     train_diverged=tdiv)); _flush()
    print(f"[P20] {protocol:13s} {regime:8s} target={target:+.2f} lam={lam:+.3f} s={sd} D={D:.4f}"+(" [B-train diverged]" if tdiv else ""))
RES=pd.DataFrame(rows)[COLS]; _flush()
print(f"\nwall clock: {(time.time()-t0)/60:.1f} min")

[resume] fresh start
50 runs to go

[P20] random-input  -0.60    target=-0.60 lam=-0.581 s=0 D=0.0000
[P20] random-input  -0.60    target=-0.60 lam=-0.584 s=1 D=0.0000
[P20] random-input  -0.60    target=-0.60 lam=-0.581 s=2 D=0.0000
[P20] random-input  -0.60    target=-0.60 lam=-0.580 s=3 D=0.0000
[P20] random-input  -0.60    target=-0.60 lam=-0.585 s=4 D=0.0000
[P20] random-input  -0.45    target=-0.45 lam=-0.477 s=0 D=0.0000
[P20] random-input  -0.45    target=-0.45 lam=-0.480 s=1 D=0.0000
[P20] random-input  -0.45    target=-0.45 lam=-0.480 s=2 D=0.0000
[P20] random-input  -0.45    target=-0.45 lam=-0.475 s=3 D=0.0000
[P20] random-input  -0.45    target=-0.45 lam=-0.480 s=4 D=0.0000
[P20] random-input  -0.30    target=-0.30 lam=-0.274 s=0 D=0.0000
[P20] random-input  -0.30    target=-0.30 lam=-0.278 s=1 D=0.0000
[P20] random-input  -0.30    target=-0.30 lam=-0.274 s=2 D=0.0000
[P20] random-input  -0.30    target=-0.30 lam=-0.276 s=3 D=0.0000
[P20] random-input  -0.30    target=-0.3

## Analysis

In [7]:
ri=RES[RES.protocol=="random-input"].groupby("target_lambda")
curve=pd.DataFrame({"achieved_lambda":ri["achieved_lambda"].mean().round(3),
                    "sync_D":ri["sync_D"].mean().round(4),
                    "sync_D_sd":ri["sync_D"].std().round(4)})
savetable(curve.reset_index(),"phase20_random_input_curve.csv")
print("RANDOM-INPUT protocol — synchronization discrepancy D across the Lyapunov axis:")
print(curve.to_string()); print()
below=curve[curve.index<0]["sync_D"].mean(); above=curve[curve.index>0]["sync_D"].mean()
print(f"  mean D for lambda<0 (stable): {below:.4f}")
print(f"  mean D for lambda>0 (chaotic): {above:.4f}")
print("  echo state property predicts D->0 below onset, D>0 above\n")
tr=RES[RES.protocol=="trained-input"].groupby("regime")
print("TRAINED-INPUT protocol — D at the three named regimes (B trained on forecast):")
for name in CFG.named:
    d=RES[(RES.protocol=="trained-input")&(RES.regime==name)]
    r=RES[(RES.protocol=="random-input")&(RES.regime==f"{CFG.named[name]:+.2f}")]
    if len(d) and len(r):
        print(f"  {name:8s} random-input D={r.sync_D.mean():.4f} -> trained-input D={d.sync_D.mean():.4f}")
print("\n  expectation: training the input drives D down (stronger synchronization),")
print("  mirroring conditional lambda -0.085 (random) -> -0.710 (trained).")
savetable(RES[RES.protocol=="trained-input"],"phase20_trained_input.csv")

saved /content/drive/My Drive/ChaoSSM/results/phase20_synchronization/phase20_random_input_curve.csv
RANDOM-INPUT protocol — synchronization discrepancy D across the Lyapunov axis:
               achieved_lambda  sync_D  sync_D_sd
target_lambda                                    
-0.60                   -0.582  0.0000     0.0000
-0.45                   -0.478  0.0000     0.0000
-0.30                   -0.276  0.0000     0.0000
-0.15                   -0.176  0.0000     0.0000
-0.02                    0.013  0.0000     0.0000
 0.15                    0.148  0.2705     0.1506
 0.30                    0.293  0.3491     0.0292

  mean D for lambda<0 (stable): 0.0000
  mean D for lambda>0 (chaotic): 0.3098
  echo state property predicts D->0 below onset, D>0 above

TRAINED-INPUT protocol — D at the three named regimes (B trained on forecast):
  ordered  random-input D=0.0000 -> trained-input D=0.0000
  edge     random-input D=0.0000 -> trained-input D=0.0000
  deep     random-input D=0.3491

## Figure (600 dpi)

In [8]:
fig,ax=plt.subplots(1,2,figsize=(12,4.6))
c=curve.sort_index()
ax[0].errorbar(c["achieved_lambda"],c["sync_D"],yerr=c["sync_D_sd"],marker="o",ms=7,lw=2,
               capsize=4,color="#2E7D32")
ax[0].axvline(0,ls=":",color="#B71C1C",lw=1.4,label=r"chaos onset ($\lambda=0$)")
ax[0].axhline(0,ls="-",color="#999",lw=0.8)
ax[0].set_xlabel(r"achieved autonomous $\lambda$"); ax[0].set_ylabel("sync discrepancy $D$")
ax[0].set_title("(a) Echo state property across the Lyapunov axis",fontsize=11.5)
ax[0].legend(fontsize=9); ax[0].grid(alpha=.25)
names=list(CFG.named); x=np.arange(len(names)); w=0.38
rvals=[RES[(RES.protocol=="random-input")&(RES.regime==f"{CFG.named[n]:+.2f}")].sync_D.mean() for n in names]
tvals=[RES[(RES.protocol=="trained-input")&(RES.regime==n)].sync_D.mean() for n in names]
ax[1].bar(x-w/2,rvals,w,label="random input",color="#9E9E9E")
ax[1].bar(x+w/2,tvals,w,label="trained input",color="#4C72B0")
ax[1].set_xticks(x); ax[1].set_xticklabels(names); ax[1].set_ylabel("sync discrepancy $D$")
ax[1].set_title("(b) Training the input strengthens synchronization",fontsize=11.5)
ax[1].legend(fontsize=9); ax[1].grid(axis="y",alpha=.25)
fig.tight_layout(); savefig(fig,"phase20_synchronization.png")

saved /content/drive/My Drive/ChaoSSM/results/phase20_synchronization/phase20_synchronization.png


## Rescue if Drive was not mounted

In [9]:
import shutil
if not RESULTS.startswith("/content/drive"):
    z=shutil.make_archive("/content/phase20_results","zip",RESULTS); print("archive:",z)
    try:
        from google.colab import files; files.download(z)
    except Exception as e: print("download",z,"manually:",e)
else: print("on Drive:",RESULTS)

on Drive: /content/drive/My Drive/ChaoSSM/results/phase20_synchronization
